In [4]:
import pandas as pd
import numpy as np

# Update this path to match your Kaggle input
df = pd.read_csv('/kaggle/input/datasets/nudratabbas/hospital-records-for-data-cleaning-medium/hospital_patients_real_world.csv')

print(df.shape)
df.head()



(5000, 7)


,PatientID,Age,Gender,Diagnosis,AdmissionDate,DischargeDate,HospitalID
0,PN-2021066,7.0,Other,Myocardial Infarction,2024-03-23,2024-03-29,HOSP-65
1,PN-4606019,36.0,Other,Pneumonia,2024-08-01,2024-08-07,HOSP-79
2,PN-2594016,70.0,Other,Influenza,2024-11-16,2024-11-23,HOSP-27
3,PN-6906914,90.0,Unknown,Acute Bronchitis,2025-07-05,2025-07-10,HOSP-64
4,PN-4656204,0.0,Female,Type 2 Diabetes,2023-08-30,2023-08-31,HOSP-31


In [5]:
def audit_report(df: pd.DataFrame) -> dict:
    """Quantify known data quality issues. Returns a dict summary."""
    report = {}

    # Missingness
    report['missing_counts'] = df[['Age', 'Gender', 'Diagnosis']].isnull().sum().to_dict()
    missing_mask = df[['Age', 'Gender', 'Diagnosis']].isnull()
    report['rows_missing_all_three'] = int((missing_mask.sum(axis=1) == 3).sum())

    # Diagnosis casing duplicates
    non_null_dx = df['Diagnosis'].dropna()
    report['diagnosis_raw_categories'] = non_null_dx.nunique()
    report['diagnosis_normalized_categories'] = non_null_dx.str.strip().str.title().nunique()

    # LOS validity
    admit = pd.to_datetime(df['AdmissionDate'])
    discharge = pd.to_datetime(df['DischargeDate'])
    los = (discharge - admit).dt.days
    report['negative_los_count'] = int((los < 0).sum())
    report['negative_los_values'] = sorted(los[los < 0].unique().tolist())

    # Duplicates
    report['duplicate_rows'] = int(df.duplicated().sum())
    report['duplicate_patient_ids'] = int(df['PatientID'].duplicated().sum())

    return report

baseline = audit_report(df)
for k, v in baseline.items():
    print(f"{k}: {v}")



missing_counts: {'Age': 350, 'Gender': 350, 'Diagnosis': 350}
rows_missing_all_three: 2
diagnosis_raw_categories: 28
diagnosis_normalized_categories: 14
negative_los_count: 150
negative_los_values: [-5]
duplicate_rows: 0
duplicate_patient_ids: 0


In [6]:
missing_mask = df[['Age', 'Gender', 'Diagnosis']].isnull()
pattern_counts = missing_mask.sum(axis=1).value_counts().sort_index()
print(pattern_counts)
# 0 = no missing fields, 1 = one field missing, 2 = two, 3 = all three

0    4005
1     942
2      51
3       2
Name: count, dtype: int64


In [7]:
def normalize_diagnosis(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize Diagnosis casing/whitespace. Preserves NaN."""
    df = df.copy()
    df['Diagnosis'] = df['Diagnosis'].str.strip().str.title()
    return df

df_clean = normalize_diagnosis(df)

print("Before:", df['Diagnosis'].nunique())
print("After:", df_clean['Diagnosis'].nunique())
df_clean['Diagnosis'].value_counts(dropna=False)

Before: 28
After: 14


Diagnosis
Urinary Tract Infection    363
Cholelithiasis             363
Osteoarthritis             352
NaN                        350
Diverticulitis             345
Atrial Fibrillation        345
Asthma                     344
Acute Bronchitis           342
Influenza                  324
Gastroenteritis            320
Hypertension               319
Myocardial Infarction      316
Chronic Kidney Disease     313
Pneumonia                  306
Type 2 Diabetes            298
Name: count, dtype: int64

In [8]:
missing_mask = df[['Age', 'Gender', 'Diagnosis']].isnull()
pattern_counts = missing_mask.sum(axis=1).value_counts().sort_index()
print(pattern_counts)

0    4005
1     942
2      51
3       2
Name: count, dtype: int64


In [10]:
def flag_missingness(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add explicit missingness flags for Age, Gender, Diagnosis rather than
    imputing — this dataset shows no reliable correlation to impute from,
    so filling values would fabricate data rather than recover it.
    """
    df = df.copy()
    for col in ['Age', 'Gender', 'Diagnosis']:
        df[f'{col}_missing'] = df[col].isnull()

    df['missing_field_count'] = df[['Age_missing', 'Gender_missing', 'Diagnosis_missing']].sum(axis=1)
    return df

df_clean = flag_missingness(df_clean)

print(df_clean['missing_field_count'].value_counts().sort_index())
print()
print("Rows missing all 3 fields (candidates for exclusion, not deletion):")
df_clean[df_clean['missing_field_count'] == 3]

missing_field_count
0    4005
1     942
2      51
3       2
Name: count, dtype: int64

Rows missing all 3 fields (candidates for exclusion, not deletion):


,PatientID,Age,Gender,Diagnosis,AdmissionDate,DischargeDate,HospitalID,Age_missing,Gender_missing,Diagnosis_missing,missing_field_count
4232,PN-8178896,NaN,NaN,NaN,2025-12-29,2026-01-07,HOSP-48,True,True,True,3
4772,PN-6739983,NaN,NaN,NaN,2024-11-16,2024-11-20,HOSP-42,True,True,True,3


In [11]:
admit = pd.to_datetime(df_clean['AdmissionDate'])
discharge = pd.to_datetime(df_clean['DischargeDate'])
los = (discharge - admit).dt.days

neg_mask = los < 0

# Hypothesis: if we swap admit/discharge for just these rows, does LOS become sane?
swapped_los = (admit[neg_mask] - discharge[neg_mask]).dt.days

print("Current negative LOS stats:")
print(los[neg_mask].describe())
print()
print("If admit/discharge were swapped for these rows, LOS would be:")
print(swapped_los.describe())

Current negative LOS stats:
count    150.0
mean      -5.0
std        0.0
min       -5.0
25%       -5.0
50%       -5.0
75%       -5.0
max       -5.0
dtype: float64

If admit/discharge were swapped for these rows, LOS would be:
count    150.0
mean       5.0
std        0.0
min        5.0
25%        5.0
50%        5.0
75%        5.0
max        5.0
dtype: float64


In [12]:
def fix_los_swap(df: pd.DataFrame) -> pd.DataFrame:
    """
    Swap AdmissionDate/DischargeDate for rows where the swap produces
    exactly a +5 day LOS (validated hypothesis: source bug swapped these
    two columns for a subset of rows). Only touches rows matching this
    exact signature — doesn't guess on other negative values if any exist.
    """
    df = df.copy()
    admit = pd.to_datetime(df['AdmissionDate'])
    discharge = pd.to_datetime(df['DischargeDate'])
    los = (discharge - admit).dt.days

    swap_mask = los == -5  # exact signature we validated

    df['LOS_was_swapped'] = swap_mask
    df.loc[swap_mask, ['AdmissionDate', 'DischargeDate']] = \
        df.loc[swap_mask, ['DischargeDate', 'AdmissionDate']].values

    return df

df_clean = fix_los_swap(df_clean)

# Recompute LOS as a real column now that dates are fixed
df_clean['AdmissionDate'] = pd.to_datetime(df_clean['AdmissionDate'])
df_clean['DischargeDate'] = pd.to_datetime(df_clean['DischargeDate'])
df_clean['LOS'] = (df_clean['DischargeDate'] - df_clean['AdmissionDate']).dt.days

print("Rows swapped:", df_clean['LOS_was_swapped'].sum())
print("Any negative LOS remaining:", (df_clean['LOS'] < 0).sum())
print(df_clean['LOS'].describe())

Rows swapped: 150
Any negative LOS remaining: 0
count    5000.000000
mean        5.452200
std         2.851481
min         1.000000
25%         3.000000
50%         5.000000
75%         8.000000
max        10.000000
Name: LOS, dtype: float64


In [13]:
# Conditions that shouldn't realistically appear in infants (Age == 0)
ADULT_ONLY_CONDITIONS = {
    'Type 2 Diabetes', 'Osteoarthritis', 'Myocardial Infarction',
    'Hypertension', 'Atrial Fibrillation', 'Chronic Kidney Disease'
}

def flag_implausible_age_diagnosis(df: pd.DataFrame) -> pd.DataFrame:
    """Flag Age == 0 rows paired with adult-only diagnoses."""
    df = df.copy()
    df['implausible_age_diagnosis'] = (
        (df['Age'] == 0) & (df['Diagnosis'].isin(ADULT_ONLY_CONDITIONS))
    )
    return df

df_clean = flag_implausible_age_diagnosis(df_clean)
print("Implausible age/diagnosis pairs flagged:", df_clean['implausible_age_diagnosis'].sum())

Implausible age/diagnosis pairs flagged: 21


In [14]:
def clean_hospital_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Full cleaning pipeline for hospital_patients_real_world.csv.

    Steps:
      1. Normalize Diagnosis casing (28 -> 14 categories)
      2. Flag (not impute) missing Age/Gender/Diagnosis
      3. Fix the validated AdmissionDate/DischargeDate swap bug (LOS == -5)
      4. Recompute LOS from corrected dates
      5. Flag implausible Age==0 + adult-only-diagnosis pairs

    Returns a cleaned dataframe with original columns preserved plus
    quality flags (*_missing, LOS_was_swapped, implausible_age_diagnosis).
    Nothing is silently dropped or imputed — flags let downstream users
    decide what to exclude for their specific analysis.
    """
    df = raw_df.copy()
    df = normalize_diagnosis(df)
    df = flag_missingness(df)
    df = fix_los_swap(df)
    df['AdmissionDate'] = pd.to_datetime(df['AdmissionDate'])
    df['DischargeDate'] = pd.to_datetime(df['DischargeDate'])
    df['LOS'] = (df['DischargeDate'] - df['AdmissionDate']).dt.days
    df = flag_implausible_age_diagnosis(df)
    return df


def scorecard(raw_df: pd.DataFrame, clean_df: pd.DataFrame) -> pd.DataFrame:
    """Before/after comparison of key data quality metrics."""
    before = audit_report(raw_df)

    rows = [
        ('Rows', len(raw_df), len(clean_df)),
        ('Diagnosis categories (raw count)', before['diagnosis_raw_categories'],
            clean_df['Diagnosis'].dropna().nunique()),
        ('Negative LOS records', before['negative_los_count'],
            int((clean_df['LOS'] < 0).sum())),
        ('Rows with LOS swap corrected', '—', int(clean_df['LOS_was_swapped'].sum())),
        ('Rows flagged: missing Age', before['missing_counts']['Age'],
            int(clean_df['Age_missing'].sum())),
        ('Rows flagged: missing Gender', before['missing_counts']['Gender'],
            int(clean_df['Gender_missing'].sum())),
        ('Rows flagged: missing Diagnosis', before['missing_counts']['Diagnosis'],
            int(clean_df['Diagnosis_missing'].sum())),
        ('Rows flagged: implausible age/diagnosis', '—',
            int(clean_df['implausible_age_diagnosis'].sum())),
        ('Duplicate rows', before['duplicate_rows'], int(clean_df.duplicated().sum())),
    ]
    return pd.DataFrame(rows, columns=['Metric', 'Before', 'After'])


# Run the full pipeline fresh from the original raw file
df_final = clean_hospital_data(df)  # df = original untouched raw load
sc = scorecard(df, df_final)
print(sc.to_string(index=False))

                                 Metric Before  After
                                   Rows   5000   5000
       Diagnosis categories (raw count)     28     14
                   Negative LOS records    150      0
           Rows with LOS swap corrected      —    150
              Rows flagged: missing Age    350    350
           Rows flagged: missing Gender    350    350
        Rows flagged: missing Diagnosis    350    350
Rows flagged: implausible age/diagnosis      —     21
                         Duplicate rows      0      0


In [15]:
def run_pipeline_tests(clean_df: pd.DataFrame):
    """Sanity checks on the cleaned dataset. Raises AssertionError if any fail."""

    # 1. No row count should be lost or duplicated
    assert len(clean_df) == 5000, f"Expected 5000 rows, got {len(clean_df)}"

    # 2. Diagnosis casing fully normalized
    dx = clean_df['Diagnosis'].dropna()
    assert dx.nunique() == 14, f"Expected 14 diagnosis categories, got {dx.nunique()}"
    assert (dx == dx.str.title()).all(), "Some diagnosis values aren't Title Case"

    # 3. No negative LOS remains
    assert (clean_df['LOS'] < 0).sum() == 0, "Negative LOS values still present"

    # 4. LOS swap count matches validated hypothesis
    assert clean_df['LOS_was_swapped'].sum() == 150, "Unexpected LOS swap count"

    # 5. Missingness flags match raw counts exactly (nothing silently dropped)
    assert clean_df['Age_missing'].sum() == 350
    assert clean_df['Gender_missing'].sum() == 350
    assert clean_df['Diagnosis_missing'].sum() == 350

    # 6. No duplicate PatientIDs introduced
    assert clean_df['PatientID'].duplicated().sum() == 0

    # 7. Implausible pairs flagged, and none of them silently correspond to Age != 0
    implausible = clean_df[clean_df['implausible_age_diagnosis']]
    assert (implausible['Age'] == 0).all(), "Implausible flag applied to non-zero age"

    print("All pipeline tests passed ✅")

run_pipeline_tests(df_final)

All pipeline tests passed ✅


In [16]:
# Export cleaned CSV
df_final.to_csv('/kaggle/working/hospital_patients_cleaned.csv', index=False)
print("Cleaned CSV saved to /kaggle/working/hospital_patients_cleaned.csv")
print(f"Shape: {df_final.shape}")

Cleaned CSV saved to /kaggle/working/hospital_patients_cleaned.csv
Shape: (5000, 14)


In [1]:
%%writefile /kaggle/working/hospital_cleaning.py
"""
Reusable cleaning pipeline for hospital_patients_real_world.csv.
Built and validated in the Project 1 notebook — see accompanying
scorecard for before/after data quality metrics.
"""
import pandas as pd

ADULT_ONLY_CONDITIONS = {
    'Type 2 Diabetes', 'Osteoarthritis', 'Myocardial Infarction',
    'Hypertension', 'Atrial Fibrillation', 'Chronic Kidney Disease'
}


def normalize_diagnosis(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Diagnosis'] = df['Diagnosis'].str.strip().str.title()
    return df


def flag_missingness(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ['Age', 'Gender', 'Diagnosis']:
        df[f'{col}_missing'] = df[col].isnull()
    df['missing_field_count'] = df[['Age_missing', 'Gender_missing', 'Diagnosis_missing']].sum(axis=1)
    return df


def fix_los_swap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    admit = pd.to_datetime(df['AdmissionDate'])
    discharge = pd.to_datetime(df['DischargeDate'])
    los = (discharge - admit).dt.days
    swap_mask = los == -5
    df['LOS_was_swapped'] = swap_mask
    df.loc[swap_mask, ['AdmissionDate', 'DischargeDate']] = \
        df.loc[swap_mask, ['DischargeDate', 'AdmissionDate']].values
    return df


def flag_implausible_age_diagnosis(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['implausible_age_diagnosis'] = (
        (df['Age'] == 0) & (df['Diagnosis'].isin(ADULT_ONLY_CONDITIONS))
    )
    return df


def clean_hospital_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Full pipeline. See module docstring for step details."""
    df = raw_df.copy()
    df = normalize_diagnosis(df)
    df = flag_missingness(df)
    df = fix_los_swap(df)
    df['AdmissionDate'] = pd.to_datetime(df['AdmissionDate'])
    df['DischargeDate'] = pd.to_datetime(df['DischargeDate'])
    df['LOS'] = (df['DischargeDate'] - df['AdmissionDate']).dt.days
    df = flag_implausible_age_diagnosis(df)
    return df

Writing /kaggle/working/hospital_cleaning.py
